# 11 · Отбор с отложенным типом требований (val + LOTO)

**Зачем** (JOURNAL, решение 2026-10-31). На val те же типы требований, что в train, поэтому отбор по val не видит
переноса: в 09 по val `linmap` (0.86) обходил `linmap_prompt` (0.74), а на новых типах — наоборот (0.64 против 0.83).

**Отбор val+loto.** Для каждого из типов deadline / quantity / budget карта подгоняется на train **без** задач
этого типа и проверяется на них — как будто тип новый. Выбирается конфигурация с максимумом среднего успеха
на val и на отложенном типе (специфичность на val — не ниже допуска). Выбор по одному val оценивается рядом
как `m@val` — так видно, что дал новый отбор.

Методы без градиента: `linmap`, `linmap_prompt`, `linmap_mix`; ориентир — LoRA. Проверка — на **свежем**
наборе переноса (seed 505). Результат **разведочный**.

In [1]:
%load_ext autoreload
%autoreload 2
import os
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")   # Mac: неподдержанные операции -> CPU

import pandas as pd
from IPython.display import Markdown, display
pd.set_option("display.precision", 3)
pd.set_option("display.width", 200)

from steering_tool_feedback.config import ExperimentConfig
from steering_tool_feedback.pipeline import Experiment
from steering_tool_feedback import journal, viz

In [ ]:
cfg = ExperimentConfig.make(
    name="loto-select-qwen3-1.7b", hypothesis="explore", notebook="notebooks/11_loto_select.ipynb",
    notes="Отбор val+loto (отложенный тип требований внутри train) для linmap, linmap_prompt, linmap_mix.",
    overrides={
        "model.name": "Qwen/Qwen3-1.7B",
        "data.n_tasks": 160,
        "data.generator": "v3",
        "eval.methods": ["baseline", "lora", "linmap", "linmap_prompt", "linmap_mix"],
        "steer.alphas": [0.25, 0.5, 1.0, 2.0, 4.0],
        "act.select": "val+loto",
        "act.ranks": [8],
        "act.alphas": [1.0, 1.5, 2.0, 3.0, 4.0],
        "episode.on_val": False,
        "transfer.seed": 505,
        "transfer.episode_methods": ["baseline", "lora", "linmap", "linmap_prompt", "linmap_mix"],
    },
)
exp = Experiment(cfg)
exp.run("fit")
exp.results_table("val")

### Подбор: val и отложенный тип

In [ ]:
f = exp.state["fitted"]
cols = ["layer", "rank", "alpha", "success", "loto_success", "score", "specificity", "loto_specificity", "spec_drop"]
for m in ("linmap", "linmap_prompt", "linmap_mix"):
    t = exp.fit_table(m)
    if t.empty:
        print(m, f[m].get("reason")); continue
    r, rv = f[m]["row"], f[m]["row_val"]
    display(Markdown(f"**{m}** — выбор val+loto: L{r['layer']}/α{r['alpha']} (score {r['score']:.3f}); "
                     f"по одному val: L{rv['layer']}/α{rv['alpha']} (score {rv['score']:.3f})"))
    display(t.sort_values("score", ascending=False)[cols].head(6).round(3))

In [ ]:
# Насколько val и отложенный тип согласны между собой по всем конфигурациям
rows = []
for m in ("linmap", "linmap_prompt", "linmap_mix"):
    t = exp.fit_table(m)
    if not t.empty:
        rows.append({"method": m, "n_configs": len(t),
                     "corr(val, loto)": t["success"].corr(t["loto_success"], method="spearman"),
                     "val_best": t["success"].max(), "loto_best": t["loto_success"].max()})
pd.DataFrame(rows).round(3)

### Свежий набор переноса (seed 505): выбор val+loto против выбора по одному val

In [ ]:
exp.transfer()
for g in ("seen", "unseen"):
    display(Markdown(f"**{g}**"))
    display(exp.results_table(f"transfer:{g}"))

In [ ]:
rows = []
for m in ("linmap", "linmap_prompt", "linmap_mix"):
    for g in ("seen", "unseen"):
        s = exp.state["summaries"][f"transfer:{g}"]
        a, b = s.get(m), s.get(f"{m}@val", s.get(m))
        if a:
            rows.append({"method": m, "group": g, "val+loto": a["success"], "val only": b["success"],
                         "same_config": f"{m}@val" not in s})
pd.DataFrame(rows)

## Вывод → журнал

In [ ]:
display(Markdown(exp.report(
    conclusion="""…""",
    next_step="""…""",
)))